## Imports

In [23]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.svm import LinearSVC

In [13]:
reviews_preprocessed = pd.read_csv("../data/processed/reviews_preprocessed.csv", index_col=0)

In [14]:
reviews_preprocessed = reviews_preprocessed.dropna(subset=["Text_clean"])

In [15]:
X = reviews_preprocessed["Text_clean"]
y = reviews_preprocessed["Sentiment"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [16]:
vectorizer = TfidfVectorizer(max_features=5000)

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

### Experiment 1: Class weighting

Testing `class_weight='balanced'` to address the class imbalance seen in the baseline
(recall 0.67 on the negative class). This penalizes errors on the minority class more heavily
during training.

In [17]:
model_balanced = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model_balanced.fit(X_train_tfidf, y_train)
y_pred_balanced = model_balanced.predict(X_test_tfidf)

print(classification_report(y_test, y_pred_balanced))

              precision    recall  f1-score   support

           0       0.60      0.89      0.72     16393
           1       0.98      0.89      0.93     88709

    accuracy                           0.89    105102
   macro avg       0.79      0.89      0.83    105102
weighted avg       0.92      0.89      0.90    105102



**Result**: Recall on the negative class improved significantly (0.67 → 0.89), but at the cost
of precision (0.83 → 0.60) — the model now over-predicts "negative". Macro F1 actually dropped
slightly (0.85 → 0.83). This is a trade-off, not a clear improvement.

### Experiment 2: N-grams

Testing bigrams (`ngram_range=(1, 2)`) to capture negation context (e.g. "not good") that gets
lost when treating words as independent tokens. Also tested increasing `max_features`
(5000 → 10000 → 20000) to see how vocabulary size affects performance with the larger n-gram space.

In [18]:
vectorizer_ngram = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
X_train_ngram = vectorizer_ngram.fit_transform(X_train)
X_test_ngram = vectorizer_ngram.transform(X_test)

model_ngram = LogisticRegression(max_iter=1000, random_state=42)
model_ngram.fit(X_train_ngram, y_train)
y_pred_ngram_5k = model_ngram.predict(X_test_ngram)

print(classification_report(y_test, y_pred_ngram_5k))

              precision    recall  f1-score   support

           0       0.85      0.70      0.77     16393
           1       0.95      0.98      0.96     88709

    accuracy                           0.93    105102
   macro avg       0.90      0.84      0.87    105102
weighted avg       0.93      0.93      0.93    105102



In [19]:
vectorizer_ngram = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_ngram = vectorizer_ngram.fit_transform(X_train)
X_test_ngram = vectorizer_ngram.transform(X_test)

model_ngram = LogisticRegression(max_iter=1000, random_state=42)
model_ngram.fit(X_train_ngram, y_train)
y_pred_ngram_10k = model_ngram.predict(X_test_ngram)

print(classification_report(y_test, y_pred_ngram_10k))

              precision    recall  f1-score   support

           0       0.87      0.72      0.79     16393
           1       0.95      0.98      0.97     88709

    accuracy                           0.94    105102
   macro avg       0.91      0.85      0.88    105102
weighted avg       0.94      0.94      0.94    105102



In [20]:
vectorizer_ngram = TfidfVectorizer(max_features=20000, ngram_range=(1, 2))
X_train_ngram = vectorizer_ngram.fit_transform(X_train)
X_test_ngram = vectorizer_ngram.transform(X_test)

model_ngram = LogisticRegression(max_iter=1000, random_state=42)
model_ngram.fit(X_train_ngram, y_train)
y_pred_ngram_20k = model_ngram.predict(X_test_ngram)

print(classification_report(y_test, y_pred_ngram_20k))

              precision    recall  f1-score   support

           0       0.88      0.74      0.80     16393
           1       0.95      0.98      0.97     88709

    accuracy                           0.94    105102
   macro avg       0.92      0.86      0.88    105102
weighted avg       0.94      0.94      0.94    105102



**Result**: Unlike class weighting, n-grams improved F1 on both classes simultaneously — no
trade-off. Macro F1 rose from 0.85 (baseline) to 0.87 (5000 features), 0.88 (10000 features),
and plateaued at 0.88 (20000 features) — confirming 10000 features as the point of diminishing
returns.

### Experiment 3: N-grams + class weighting combined

Testing whether combining both techniques compensates for class weighting's precision loss.

In [24]:
vectorizer_final = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_final = vectorizer_final.fit_transform(X_train)
X_test_final = vectorizer_final.transform(X_test)

model_final = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model_final.fit(X_train_final, y_train)
y_pred_final = model_final.predict(X_test_final)



print(classification_report(y_test, y_pred_final))

              precision    recall  f1-score   support

           0       0.67      0.91      0.77     16393
           1       0.98      0.92      0.95     88709

    accuracy                           0.91    105102
   macro avg       0.82      0.91      0.86    105102
weighted avg       0.93      0.91      0.92    105102

              precision    recall  f1-score   support

           0       0.86      0.76      0.81     16393
           1       0.96      0.98      0.97     88709

    accuracy                           0.94    105102
   macro avg       0.91      0.87      0.89    105102
weighted avg       0.94      0.94      0.94    105102



**Result**: Combining both techniques performed worse (macro F1 0.86) than n-grams alone (0.88)
— class weighting's precision trade-off persists even with the improved n-gram features.

### Experiment 4: LinearSVC
Testing `LinearSVC` instead of Logistic Regression on the same feature set (n-grams, 10000
features), since SVMs sometimes perform better on sparse, high-dimensional TF-IDF data due to
their margin-based optimization approach rather than probabilistic loss minimization.

In [25]:
model_svc = LinearSVC(max_iter=1000, random_state=42)
model_svc.fit(X_train_final, y_train)
y_pred_svc = model_svc.predict(X_test_final)

print(classification_report(y_test, y_pred_svc))

              precision    recall  f1-score   support

           0       0.86      0.76      0.81     16393
           1       0.96      0.98      0.97     88709

    accuracy                           0.94    105102
   macro avg       0.91      0.87      0.89    105102
weighted avg       0.94      0.94      0.94    105102



**Result**: LinearSVC outperformed Logistic Regression on the same features — macro F1 improved
from 0.88 to 0.89, with gains in both precision and recall on the negative class (no trade-off).



### Final model selection

**Selected model: LinearSVC on TF-IDF with bigrams (`max_features=10000`, `ngram_range=(1, 2)`).**
This achieved the best macro F1 (0.89) across all tested configurations — Negative class:
precision 0.86, recall 0.76, F1 0.81. Positive class: precision 0.96, recall 0.98, F1 0.97.